In [ ]:
#matmul & transpose on the fly using tiling

#include<iostream>
#include<cstdio>
#include<cstdlib>
#include<sys/time.h>
#include<cuda.h>
using namespace std;

__global__ void transpose(int a[], int b[], int r, int c) {
    __shared__ int tile[32][33];

    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < r && col < c) {
        tile[threadIdx.y][threadIdx.x] = a[row * c + col];
    }

    __syncthreads();

    int new_col = blockIdx.y * blockDim.x + threadIdx.x;
    int new_row = blockIdx.x * blockDim.y + threadIdx.y;

    if (new_row < c && new_col < r) {
        b[new_row * r + new_col] = tile[threadIdx.x][threadIdx.y];
    }
}

__global__ void matmul(int a[], int d[], int b[], int c[], int e[], int p, int q, int r) {
	__shared__ int tileA[32][33];
	__shared__ int tileB[32][33];
	__shared__ int tileC[32][33];
	__shared__ int tileD[32][33];

	int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
	int sum = 0;

	for(int i=0;i<(q+31)/32;i++) {

		if(blockIdx.y*32 + threadIdx.y < p && i*32 + threadIdx.x < q) {
			tileA[threadIdx.y][threadIdx.x] = a[i*32*p + threadIdx.x*p + blockIdx.y*32 + threadIdx.y];
		}
		else {
			tileA[threadIdx.y][threadIdx.x] = 0;
		}

		if(i*32 + threadIdx.x < q && blockIdx.y*32 + threadIdx.y < p) {
			tileC[threadIdx.y][threadIdx.x] = c[blockIdx.y*32*q + threadIdx.y*q + i*32 + threadIdx.x];
		}
		else {
			tileC[threadIdx.y][threadIdx.x] = 0;
		}

		if(blockIdx.x*32 + threadIdx.x < r && i*32 + threadIdx.y < q) {
			tileB[threadIdx.y][threadIdx.x] = b[i*32*r + threadIdx.y*r + blockIdx.x*32 + threadIdx.x];
		}
		else {
			tileB[threadIdx.y][threadIdx.x] = 0;
		}

		if(i*32 + threadIdx.y < q && blockIdx.x*32 + threadIdx.x < r) {
			tileD[threadIdx.y][threadIdx.x] = d[blockIdx.x*32*q + threadIdx.x*q + i*32 + threadIdx.y];
		}
		else {
			tileD[threadIdx.y][threadIdx.x] = 0;
		}

		__syncthreads();

		for(int j=0;j<32;j++) {
			sum+=(tileA[threadIdx.y][j]*tileB[j][threadIdx.x])+(tileC[threadIdx.y][j]*tileD[j][threadIdx.x]);
		}

		__syncthreads();

	}

	if(row < p && col < r) {
		e[row * r + col] = sum;
	}
}


// function to compute the output matrix
void compute(int p, int q, int r, int *h_matrixA, int *h_matrixB,
	         int *h_matrixC, int *h_matrixD, int *h_matrixE){
	// Device variables declarations...
	int *d_matrixA, *d_matrixB, *d_matrixC, *d_matrixD, *d_matrixE;

	// allocate memory...
	cudaMalloc(&d_matrixA, q * p * sizeof(int));
	cudaMalloc(&d_matrixB, q * r * sizeof(int));
	cudaMalloc(&d_matrixC, p * q * sizeof(int));
	cudaMalloc(&d_matrixD, r * q * sizeof(int));
	cudaMalloc(&d_matrixE, p * r * sizeof(int));

	// copy the values...
	cudaMemcpy(d_matrixA, h_matrixA, q * p * sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_matrixB, h_matrixB, q * r * sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_matrixC, h_matrixC, p * q * sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_matrixD, h_matrixD, r * q * sizeof(int), cudaMemcpyHostToDevice);

	/* ****************************************************************** */
	/* Write your code here */
	/* Configure and launch kernels */
	dim3 threadsPerBlock(32,32);
	dim3 numBlocksMul((r+31)/32, (p+31)/32);
	matmul<<<numBlocksMul, threadsPerBlock>>>(d_matrixA, d_matrixD, d_matrixB, d_matrixC, d_matrixE, p, q, r);

	/* ****************************************************************** */

	// copy the result back...
	cudaMemcpy(h_matrixE, d_matrixE, p * r * sizeof(int), cudaMemcpyDeviceToHost);

	// deallocate the memory...
	cudaFree(d_matrixA);
	cudaFree(d_matrixB);
	cudaFree(d_matrixC);
	cudaFree(d_matrixD);
	cudaFree(d_matrixE);
}

// function to read the input matrices from the input file
void readMatrix(FILE *inputFilePtr, int *matrix, int rows, int cols) {
	for(int i=0; i<rows; i++) {
		for(int j=0; j<cols; j++) {
			fscanf(inputFilePtr, "%d", &matrix[i*cols+j]);
		}
	}
}

// function to write the output matrix into the output file
void writeMatrix(FILE *outputFilePtr, int *matrix, int rows, int cols) {
	for(int i=0; i<rows; i++) {
		for(int j=0; j<cols; j++) {
			fprintf(outputFilePtr, "%d ", matrix[i*cols+j]);
		}
		fprintf(outputFilePtr, "\n");
	}
}



int main(int argc, char **argv) {
	// variable declarations
	int p, q, r;
	int *matrixA, *matrixB, *matrixC, *matrixD, *matrixE;
	struct timeval t1, t2;
	double seconds, microSeconds;

	// get file names from command line
	char *inputFileName = argv[1];
	char *outputFileName = argv[2];

	// file pointers
	FILE *inputFilePtr, *outputFilePtr;

    inputFilePtr = fopen(inputFileName, "r");
	if(inputFilePtr == NULL) {
	    printf("Failed to open the input file.!!\n");
		return 0;
	}

	// read input values
	fscanf(inputFilePtr, "%d %d %d", &p, &q, &r);

	// allocate memory and read input matrices
	matrixA = (int*) malloc(q * p * sizeof(int));
	matrixB = (int*) malloc(q * r * sizeof(int));
	matrixC = (int*) malloc(p * q * sizeof(int));
	matrixD = (int*) malloc(r * q * sizeof(int));
	readMatrix(inputFilePtr, matrixA, q, p);
	readMatrix(inputFilePtr, matrixB, q, r);
	readMatrix(inputFilePtr, matrixC, p, q);
	readMatrix(inputFilePtr, matrixD, r, q);

	// allocate memory for output matrix
	matrixE = (int*) malloc(p * r * sizeof(int));

	// call the compute function
	gettimeofday(&t1, NULL);
	compute(p, q, r, matrixA, matrixB, matrixC, matrixD, matrixE);
	cudaDeviceSynchronize();
	gettimeofday(&t2, NULL);

	// print the time taken by the compute function
	seconds = t2.tv_sec - t1.tv_sec;
	microSeconds = t2.tv_usec - t1.tv_usec;
	printf("Time taken (ms): %.3f\n", 1000*seconds + microSeconds/1000);

	// store the result into the output file
	outputFilePtr = fopen(outputFileName, "w");
	writeMatrix(outputFilePtr, matrixE, p, r);

	// close files
	fclose(inputFilePtr);
	fclose(outputFilePtr);

	// deallocate memory
	free(matrixA);
	free(matrixB);
	free(matrixC);
	free(matrixD);
	free(matrixE);

	return 0;
}

In [ ]:
#matmul with separate transpose kernel for coalesced access

#include<iostream>
#include<cstdio>
#include<cstdlib>
#include<sys/time.h>
#include<cuda.h>
using namespace std;

__global__ void transpose(int a[], int b[], int r, int c) {
    __shared__ int tile[32][33];

    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < r && col < c) {
        tile[threadIdx.y][threadIdx.x] = a[row * c + col];
    }

    __syncthreads();

    int new_col = blockIdx.y * blockDim.x + threadIdx.x;
    int new_row = blockIdx.x * blockDim.y + threadIdx.y;

    if (new_row < c && new_col < r) {
        b[new_row * r + new_col] = tile[threadIdx.x][threadIdx.y];
    }
}

__global__ void matmul(int at[], int dt[], int b[], int c[], int e[], int p, int q, int r) {
	__shared__ int tileA[32][33];
	__shared__ int tileB[32][33];
	__shared__ int tileC[32][33];
	__shared__ int tileD[32][33];

	int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
	int sum = 0;

	for(int i=0;i<(q+31)/32;i++) {

		if(i*32 + threadIdx.x < q && blockIdx.y*32 + threadIdx.y < p) {
			tileA[threadIdx.y][threadIdx.x] = at[blockIdx.y*32*q + threadIdx.y*q + i*32 + threadIdx.x];
			tileC[threadIdx.y][threadIdx.x] = c[blockIdx.y*32*q + threadIdx.y*q + i*32 + threadIdx.x];
		}
		else {
			tileA[threadIdx.y][threadIdx.x] = 0;
			tileC[threadIdx.y][threadIdx.x] = 0;
		}

		if(blockIdx.x*32 + threadIdx.x < r && i*32 + threadIdx.y < q) {
			tileD[threadIdx.y][threadIdx.x] = dt[i*32*r + threadIdx.y*r + blockIdx.x*32 + threadIdx.x];
			tileB[threadIdx.y][threadIdx.x] = b[i*32*r + threadIdx.y*r + blockIdx.x*32 + threadIdx.x];
		}
		else {
			tileB[threadIdx.y][threadIdx.x] = 0;
			tileD[threadIdx.y][threadIdx.x] = 0;
		}

		__syncthreads();

		for(int j=0;j<32;j++) {
			sum+=(tileA[threadIdx.y][j]*tileB[j][threadIdx.x])+(tileC[threadIdx.y][j]*tileD[j][threadIdx.x]);
		}

		__syncthreads();

	}

	if(row < p && col < r) {
		e[row * r + col] = sum;
	}
}


// function to compute the output matrix
void compute(int p, int q, int r, int *h_matrixA, int *h_matrixB,
	         int *h_matrixC, int *h_matrixD, int *h_matrixE){
	// Device variables declarations...
	int *d_matrixA, *d_matrixB, *d_matrixC, *d_matrixD, *d_matrixE;

	// allocate memory...
	cudaMalloc(&d_matrixA, q * p * sizeof(int));
	cudaMalloc(&d_matrixB, q * r * sizeof(int));
	cudaMalloc(&d_matrixC, p * q * sizeof(int));
	cudaMalloc(&d_matrixD, r * q * sizeof(int));
	cudaMalloc(&d_matrixE, p * r * sizeof(int));

	// copy the values...
	cudaMemcpy(d_matrixA, h_matrixA, q * p * sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_matrixB, h_matrixB, q * r * sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_matrixC, h_matrixC, p * q * sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_matrixD, h_matrixD, r * q * sizeof(int), cudaMemcpyHostToDevice);

	/* ****************************************************************** */
	/* Write your code here */
	/* Configure and launch kernels */
	int *d_matrixAT, *d_matrixDT;
	dim3 threadsPerBlock(32,32);

	cudaMalloc(&d_matrixAT, p * q * sizeof(int));
	dim3 numBlocksA((p+31)/32, (q+31)/32);
	transpose<<<numBlocksA, threadsPerBlock>>>(d_matrixA, d_matrixAT, q, p);

	cudaMalloc(&d_matrixDT, q * r * sizeof(int));
	dim3 numBlocksD((q+31)/32, (r+31)/32);
	transpose<<<numBlocksD, threadsPerBlock>>>(d_matrixD, d_matrixDT, r, q);

	dim3 numBlocksMul((r+31)/32, (p+31)/32);
	matmul<<<numBlocksMul, threadsPerBlock>>>(d_matrixAT, d_matrixDT, d_matrixB, d_matrixC, d_matrixE, p, q, r);

	/* ****************************************************************** */

	// copy the result back...
	cudaMemcpy(h_matrixE, d_matrixE, p * r * sizeof(int), cudaMemcpyDeviceToHost);

	// deallocate the memory...
	cudaFree(d_matrixA);
	cudaFree(d_matrixB);
	cudaFree(d_matrixC);
	cudaFree(d_matrixD);
	cudaFree(d_matrixE);
	cudaFree(d_matrixAT);
	cudaFree(d_matrixDT);
}

// function to read the input matrices from the input file
void readMatrix(FILE *inputFilePtr, int *matrix, int rows, int cols) {
	for(int i=0; i<rows; i++) {
		for(int j=0; j<cols; j++) {
			fscanf(inputFilePtr, "%d", &matrix[i*cols+j]);
		}
	}
}

// function to write the output matrix into the output file
void writeMatrix(FILE *outputFilePtr, int *matrix, int rows, int cols) {
	for(int i=0; i<rows; i++) {
		for(int j=0; j<cols; j++) {
			fprintf(outputFilePtr, "%d ", matrix[i*cols+j]);
		}
		fprintf(outputFilePtr, "\n");
	}
}



int main(int argc, char **argv) {
	// variable declarations
	int p, q, r;
	int *matrixA, *matrixB, *matrixC, *matrixD, *matrixE;
	struct timeval t1, t2;
	double seconds, microSeconds;

	// get file names from command line
	char *inputFileName = argv[1];
	char *outputFileName = argv[2];

	// file pointers
	FILE *inputFilePtr, *outputFilePtr;

    inputFilePtr = fopen(inputFileName, "r");
	if(inputFilePtr == NULL) {
	    printf("Failed to open the input file.!!\n");
		return 0;
	}

	// read input values
	fscanf(inputFilePtr, "%d %d %d", &p, &q, &r);

	// allocate memory and read input matrices
	matrixA = (int*) malloc(q * p * sizeof(int));
	matrixB = (int*) malloc(q * r * sizeof(int));
	matrixC = (int*) malloc(p * q * sizeof(int));
	matrixD = (int*) malloc(r * q * sizeof(int));
	readMatrix(inputFilePtr, matrixA, q, p);
	readMatrix(inputFilePtr, matrixB, q, r);
	readMatrix(inputFilePtr, matrixC, p, q);
	readMatrix(inputFilePtr, matrixD, r, q);

	// allocate memory for output matrix
	matrixE = (int*) malloc(p * r * sizeof(int));

	// call the compute function
	gettimeofday(&t1, NULL);
	compute(p, q, r, matrixA, matrixB, matrixC, matrixD, matrixE);
	cudaDeviceSynchronize();
	gettimeofday(&t2, NULL);

	// print the time taken by the compute function
	seconds = t2.tv_sec - t1.tv_sec;
	microSeconds = t2.tv_usec - t1.tv_usec;
	printf("Time taken (ms): %.3f\n", 1000*seconds + microSeconds/1000);

	// store the result into the output file
	outputFilePtr = fopen(outputFileName, "w");
	writeMatrix(outputFilePtr, matrixE, p, r);

	// close files
	fclose(inputFilePtr);
	fclose(outputFilePtr);

	// deallocate memory
	free(matrixA);
	free(matrixB);
	free(matrixC);
	free(matrixD);
	free(matrixE);

	return 0;
}